In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import h5py
from sklearn.model_selection import train_test_split

In [ ]:
%%capture
import keras
import tensorflow.keras as tk
import tensorflow.keras.backend as K
from tensorflow.keras.layers import LeakyReLU
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# from IPython.display import SVG
# from keras.utils.vis_utils import model_to_dot

from time import time
import numpy as np

In [ ]:
dataset_file = h5py.File("/kaggle/input/datasets/pinxau1000/radioml2018/GOLD_XYZ_OSC.0001_1024.hdf5","r")

In [ ]:
base_modulation_clases = ['OOK',
 '4ASK',
 '8ASK',
 'BPSK',
 'QPSK',
 '8PSK',
 '16PSK',
 '32PSK',
 '16APSK',
 '32APSK',
 '64APSK',
 '128APSK',
 '16QAM',
 '32QAM',
 '64QAM',
 '128QAM',
 '256QAM',
 'AM-SSB-WC',
 'AM-SSB-SC',
 'AM-DSB-WC',
 'AM-DSB-SC',
 'FM',
 'GMSK',
 'OQPSK']

#"Define the classes (set the label for each class). They must be ordered exactly as specified in the provided dataset; the order cannot be swapped."

selected_modulation_classes = [
#                               '8ASK', 
                               '4ASK', 
                               'BPSK', 
                               'QPSK', 
                               '16PSK', 
                               '16QAM', 
#                                '64QAM', 
#                                '256QAM',
                                'FM',
                                'AM-DSB-WC',
                                '32APSK'
                              ]
## This code is used to find the positions (or indices) of the "selected" modulations (selected_modulation_classes) within the main list (base_modulation_clases) and store them in the variable selected_classes_id.

selected_classes_id = list()
for selected_class in selected_modulation_classes:
    for id, base_class in enumerate(base_modulation_clases):
        if selected_class == base_class:
            selected_classes_id.append(id)
selected_classes_id


In [ ]:
"Values in the dataset are sorted from highest to lowest."
SNRs_in_dataset = [ 20, 18, 16, 14, 12, 10, 8, 6, 4, 2, 0, -2, -4, -6, -8, -10 , -20]
desired_SNRs = [ -10 ]  
samples_per_snr = 4096
samples_per_class = 106496

X_data = []
y_data = []

for id in selected_classes_id:
    for snr in desired_SNRs:  
        snr_index = SNRs_in_dataset.index(snr)
        start = id * samples_per_class + snr_index * samples_per_snr
        end = start + samples_per_snr

        X_data.append(dataset_file['X'][start:end])
        
        y_data.append(dataset_file['Y'][start:end])

X_data = np.concatenate(X_data)
y_data = np.concatenate(y_data)


In [ ]:
X_data = X_data.reshape(len(X_data), 32, 32, 2)
X_data.shape

In [ ]:
y_data_df = pd.DataFrame(y_data)
for column in y_data_df.columns:
    if sum(y_data_df[column]) == 0:
        y_data_df = y_data_df.drop(columns = [column])
y_data_df.columns = selected_modulation_classes
y_data_df

#y_data_df.loc[y_data_df['BPSK']==1]

In [ ]:
from collections import Counter
import numpy as np

"Convert y_data from one-hot encoding to class labels (the index of the '1' in each row)."
y_data_labels = np.argmax(y_data, axis=1)

# Check the number of data points in y_data_labels
class_counts = Counter(y_data_labels)

# Display the number of samples for each modulation class.
for modulation_class, count in class_counts.items():
    print(f"Modulation {modulation_class}: {count} samples")

In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data_df, test_size = 0.2,random_state = 42)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, initializers

class S2MLayer(tf.keras.layers.Layer):
    def __init__(self, slice_size):
        super(S2MLayer, self).__init__()
        self.k = slice_size  # window size

    def build(self, input_shape):
        self.N = input_shape[-1]  # signal length
        self.filter_I = self.add_weight(
            shape=(self.k, self.k),
            initializer='glorot_uniform',
            trainable=True,
            name='filter_I'
        )
        self.filter_Q = self.add_weight(
            shape=(self.k, self.k),
            initializer='glorot_uniform',
            trainable=True,
            name='filter_Q'
        )

    def call(self, inputs):
        # inputs shape: (batch, 2, N)
        I = inputs[:, 0, :]  # (batch, N)
        Q = inputs[:, 1, :]  # (batch, N)

        def compute_matrix(x, filt):
            slices = tf.signal.frame(x, self.k, 1, axis=-1)  # shape (batch, N-k+1, k)
            slices_T = tf.transpose(slices, perm=[0, 2, 1])  # shape (batch, k, N-k+1)
            left = tf.matmul(slices, filt)                  # shape (batch, N-k+1, k)
            result = tf.matmul(left, slices_T)              # shape (batch, N-k+1, N-k+1)
            return result

        MI = compute_matrix(I, self.filter_I)
        MQ = compute_matrix(Q, self.filter_Q)

        M = tf.stack([MI, MQ], axis=-1)  # (batch, N−k+1, N−k+1, 2)
        return M


In [ ]:
def conv1d_block(x, filters, kernel_size=3, strides=1):
    x = layers.Conv1D(filters, kernel_size, strides=strides, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def identity_block_2d(x, filters, kernel_size=3):
    shortcut = x
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, shortcut])
    x = layers.ReLU()(x)
    return x

def conv_block_2d(x, filters, kernel_size=3, strides=2):
    shortcut = layers.Conv2D(filters, 1, strides=strides, padding='same')(x)
    shortcut = layers.BatchNormalization()(shortcut)

    x = layers.Conv2D(filters, kernel_size, strides=strides, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)

    x = layers.add([x, shortcut])
    x = layers.ReLU()(x)
    return x


In [ ]:
from tensorflow.keras import layers, models

def build_signet2d_head(input_shape=(32, 32, 2), num_classes=11):
    input_layer = layers.Input(shape=input_shape)

    x = conv_block_2d(input_layer, 64)
    x = identity_block_2d(x, 64)
    x = conv_block_2d(x, 128)
    x = identity_block_2d(x, 128)
    x = conv_block_2d(x, 256)
    x = identity_block_2d(x, 256)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs=input_layer, outputs=output, name="SigNet2.0-Head")


In [ ]:
model = build_signet2d_head(input_shape=(32, 32, 2), num_classes=8)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
import tensorflow.keras as tk

batch_size = 64
epochs = 30

# สร้าง callbacks
path_checkpoint = "model_checkpoint.weights.h5"
es_callback = tk.callbacks.EarlyStopping(
    monitor="val_accuracy",  # ใช้ validation accuracy
    patience=10,
    restore_best_weights=True,
    verbose=1
)

modelckpt_callback = tk.callbacks.ModelCheckpoint(
    monitor="val_accuracy",
    filepath=path_checkpoint,
    verbose=1,
    save_weights_only=True,
    save_best_only=True,
)

# เทรนโมเดล
history = model.fit(
    x=X_train,                  # Shape: (batch, 512, 2)
    y=y_train,                  # One-hot encoded labels (batch, num_classes)
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(X_test, y_test),
    callbacks=[es_callback, modelckpt_callback]
)


In [ ]:
plt.plot(np.c_[history.history['accuracy'],history.history['val_accuracy']])
plt.legend(['accuracy', 'val_accuracy'])

In [ ]:
plt.plot(np.c_[history.history['loss'],history.history['val_loss']])
plt.legend(['loss', 'val_loss'])

In [ ]:
model_predictions = model.predict(X_test)
model_predictions

In [ ]:
def convert_to_matrix(logit_list):
    logit_list = list(logit_list)
    return logit_list.index(max(logit_list))

cm = confusion_matrix(
    y_true = list(map(convert_to_matrix , y_test.values)),
    y_pred = list(map(convert_to_matrix , model_predictions)),
)
disp = ConfusionMatrixDisplay(confusion_matrix = cm,
                              display_labels=selected_modulation_classes)
disp.plot()
plt.show()
